In [1]:
from validator import validate_yaml
from logger_setup import setup_logging
from loader import download_from_github, parse_excel_file
from transformer import apply_transformations
from saver import save_output
from db import log_to_db, log_message_to_db, load_to_postgres
import time
import os

config = validate_yaml('yaml_file.yml')
logger = setup_logging(config)

run_id = int(time.time())
params = config.conn_type.param

log_to_db('STARTED', 'START', run_id, config.tables.log, params)

try:
    if config.conn_type.type == 'hybrid':
        local_file = params['local_file']
        if not download_from_github(params['url'], local_file):
            if not os.path.exists(local_file):
                raise FileNotFoundError(f"File not found: {local_file}")
        
        df = parse_excel_file(local_file, config.parse.rows)
        df_filtered = df[config.columns]
        df_transformed, _ = apply_transformations(df_filtered, config)
        
        save_output(df_transformed, config.output)
        load_to_postgres(df_transformed, params, 'excel_data')
        
        print(df_transformed)
        
except Exception as e:
    logger.error(f"Error: {e}")
    log_to_db('ERROR', 'FAILED', run_id, config.tables.log, params, error=str(e))

[2026-07-03 06:38:18,493] INFO: Attempting to download file from GitHub
[2026-07-03 06:38:18,994] INFO: File downloaded from GitHub: Book.xlsx
[2026-07-03 06:38:18,997] INFO: Parsing Excel file
[2026-07-03 06:38:19,376] INFO: Selected columns: ['fio', 'summa', 'chel', 'sheet_name']
[2026-07-03 06:38:19,377] INFO: Renamed column 'fio' to 'ФИО'
[2026-07-03 06:38:19,378] INFO: Renamed column 'summa' to 'Сумма'
[2026-07-03 06:38:19,378] INFO: Renamed column 'chel' to 'Чел'
[2026-07-03 06:38:19,380] INFO: Renamed column 'sheet_name' to 'Название_листа'
[2026-07-03 06:38:19,468] INFO: Created table excel_data with columns: ['ФИО', 'Сумма', 'Чел', 'Название_листа']


Data exported to CSV: output_data.csv
                              ФИО      Сумма  Чел Название_листа
1            Иванов Иван Иванович   55000.00    1         Лист 1
2          Петрова Анна Сергеевна   67500.00    1         Лист 1
3      Смирнов Дмитрий Алексеевич  123000.00    3         Лист 1
4    Кузнецова Ольга Владимировна   89200.00    2         Лист 1
5         Попов Андрей Викторович   34800.00    1         Лист 1
6   Соколова Екатерина Дмитриевна   56300.00    2         Лист 2
7    Васильев Павел Александрович   92750.00    4         Лист 2
8      Михайлова Ирина Викторовна   41200.00    1         Лист 2
9         Новиков Денис Сергеевич   78400.00    3         Лист 2
10     Федорова Наталья Андреевна  104500.00    5         Лист 2
11     Григорьев Евгений Олегович   37450.50    1         Лист 3
12         Зайцева Мария Петровна   63280.75    2         Лист 3
13     Морозов Александр Игоревич   88000.00    3         Лист 3
14      Волкова Юлия Владимировна   51999.99    2   